# 使用 Pinecone 实现检索增强生成

本笔记本演示如何通过名为检索增强生成（RAG）的技术，把 Claude 与你的 Pinecone 向量数据库中的数据连接起来。我们将涵盖以下步骤：

1. 使用 Voyage AI 的嵌入模型对数据集做嵌入向量
2. 把嵌入向量上传到 Pinecone 索引
3. 从向量数据库中检索信息
4. 使用 Claude 结合数据库中的信息回答问题

## 环境准备
首先，安装本笔记本所需的库并设置所需的 API 密钥。我们需要获取一个 [Claude API 密钥](https://docs.claude.com/claude/reference/getting-started-with-the-api)、一个免费的 [Pinecone API 密钥](https://docs.pinecone.io/docs/quickstart)，以及一个免费的 [Voyage AI API 密钥](https://docs.voyageai.com/install/)。

In [ ]:
%pip install anthropic datasets pinecone-client voyageai

In [ ]:
# Insert your API keys here
ANTHROPIC_API_KEY = "<YOUR_ANTHROPIC_API_KEY>"
PINECONE_API_KEY = "<YOUR_PINECONE_API_KEY>"
VOYAGE_API_KEY = "<YOUR_VOYAGE_API_KEY>"

## 下载数据集
现在下载 Amazon 产品数据集，它包含超过 1 万条 Amazon 产品描述，并将其载入 DataFrame。

In [ ]:
import pandas as pd

# Download the JSONL file
!wget  https://www-cdn.anthropic.com/48affa556a5af1de657d426bcc1506cdf7e2f68e/amazon-products.jsonl

data = []
with open("amazon-products.jsonl") as file:
    for line in file:
        try:
            data.append(eval(line))  # noqa: S307
        except (SyntaxError, ValueError):
            # Skip malformed lines in the dataset
            pass

df = pd.DataFrame(data)
display(df.head())
len(df)

## 向量数据库

要创建向量数据库，首先需要 Pinecone 的免费 API 密钥。拿到密钥后，按如下方式初始化数据库：

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

接下来设置索引规格，用于定义我们要在哪个云服务商和区域部署索引。所有可用的服务商和区域列在[这里](https://www.pinecone.io/docs/data-types/metadata/)。


In [ ]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(cloud="aws", region="us-west-2")

然后初始化索引。我们将使用 Voyage 的 “voyage-2” 模型生成嵌入向量，因此把维度设为 1024。

In [ ]:
import time

index_name = "amazon-products"
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1024,  # dimensionality of voyage-2 embeddings
        metric="dotproduct",
        spec=spec,
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

可以看到新建的 Pinecone 索引 total_vector_count 为 0，因为我们还没有添加任何向量。

## 嵌入向量
要开始使用 Voyage 的嵌入向量，请[到这里](https://www.voyageai.com)获取 API 密钥。

现在设置 Voyage 客户端，并演示如何用 `embed` 方法创建嵌入向量。要了解 Voyage 嵌入向量与 Claude 的配合使用，见[这份笔记本](https://github.com/anthropics/anthropic-cookbook/blob/main/third_party/VoyageAI/how_to_create_embeddings.md)。

In [ ]:
import voyageai

vo = voyageai.Client(api_key=VOYAGE_API_KEY)

texts = ["Sample text 1", "Sample text 2"]

result = vo.embed(texts, model="voyage-2", input_type="document")
print(result.embeddings[0])
print(result.embeddings[1])

## 把数据上传到 Pinecone 索引

嵌入向量模型设置完成后，就可以对产品描述做嵌入，并把嵌入向量上传到 Pinecone 索引。

In [ ]:
from time import sleep

from tqdm.auto import tqdm

descriptions = df["text"].tolist()
batch_size = 100  # how many embeddings we create and insert at once

for i in tqdm(range(0, len(descriptions), batch_size)):
    # find end of batch
    i_end = min(len(descriptions), i + batch_size)
    descriptions_batch = descriptions[i:i_end]
    # create embeddings (try-except added to avoid RateLimitError. Voyage currently allows 300/requests per minute.)
    done = False
    while not done:
        try:
            res = vo.embed(descriptions_batch, model="voyage-2", input_type="document")
            done = True
        except Exception:
            sleep(5)

    embeds = [record for record in res.embeddings]
    # create unique IDs for each text
    ids_batch = [f"description_{idx}" for idx in range(i, i_end)]

    # Create metadata dictionaries for each text
    metadata_batch = [{"description": description} for description in descriptions_batch]

    to_upsert = list(zip(ids_batch, embeds, metadata_batch, strict=False))

    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

## 执行查询

索引填充完成后，就可以开始查询获取结果。把一个自然语言问题做成嵌入向量，再对索引执行查询，即可返回语义相近的产品描述。

In [49]:
USER_QUESTION = (
    "I want to get my daughter more interested in science. What kind of gifts should I get her?"
)

question_embed = vo.embed([USER_QUESTION], model="voyage-2", input_type="query")
results = index.query(vector=question_embed.embeddings, top_k=5, include_metadata=True)
results

{'matches': [{'id': 'description_1771',
              'metadata': {'description': 'Product Name: Scientific Explorer '
                                          'My First Science Kids Science '
                                          'Experiment Kit\n'
                                          '\n'
                                          'About Product: Experiments to spark '
                                          'creativity and curiosity | Grow '
                                          'watery crystals, create a rainbow '
                                          'in a plate, explore the science of '
                                          'color and more | Represents STEM '
                                          '(Science, Technology, Engineering, '
                                          'Math) principles – open ended toys '
                                          'to construct, engineer, explorer '
                                          'and experiment | Inclu

## 优化搜索

这些结果不错，但还能进一步优化。借助 Claude，可以从用户的问题生成搜索关键词。这样就能对索引进行广泛多样的搜索，获得更相关的产品描述。

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)


def get_completion(prompt):
    completion = client.completions.create(
        model="claude-2.1",
        prompt=prompt,
        max_tokens_to_sample=1024,
    )
    return completion.completion

In [ ]:
def create_keyword_prompt(question):
    return f"""\n\nHuman: Given a question, generate a list of 5 very diverse search keywords that can be used to search for products on Amazon.

The question is: {question}

Output your keywords as a JSON that has one property "keywords" that is a list of strings. Only output valid JSON.\n\nAssistant:{{"""

Anthropic 客户端设置好、提示词创建好之后，就可以开始从问题中生成关键词。我们把关键词以 JSON 对象输出，便于从 Claude 的输出中解析。

In [ ]:
keyword_json = "{" + get_completion(create_keyword_prompt(USER_QUESTION))
print(keyword_json)

In [ ]:
import json

# Extract the keywords from the JSON
data = json.loads(keyword_json)
keywords_list = data["keywords"]
print(keywords_list)

关键词现在已在列表中，接下来对每个关键词做嵌入，对索引执行查询，并返回最相关的前 3 条产品描述。

In [ ]:
results_list = []
for keyword in keywords_list:
    # get the embeddings for the keywords
    query_embed = vo.embed([keyword], model="voyage-2", input_type="query")
    # search for the embeddings in the Pinecone index
    search_results = index.query(vector=query_embed.embeddings, top_k=3, include_metadata=True)
    # append the search results to the list
    for search_result in search_results.matches:
        results_list.append(search_result["metadata"]["description"])
print(len(results_list))

## 用 Claude 回答

现在有了产品描述列表，把它们格式化为 Claude 训练时使用的搜索模板，并把格式化后的描述传入另一个提示词。

In [47]:
# Formatting search results
def format_results(extracted: list[str]) -> str:
    result = "\n".join(
        [
            f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
            for i, r in enumerate(extracted)
        ]
    )

    return f"\n<search_results>\n{result}\n</search_results>"


def create_answer_prompt(results_list, question):
    return f"""\n\nHuman: {format_results(results_list)} Using the search results provided within the <search_results></search_results> tags, please answer the following question <question>{question}</question>. Do not reference the search results in your answer.\n\nAssistant:"""

最后，提出原始用户的问题，并从 Claude 获得答案。

In [50]:
answer = get_completion(create_answer_prompt(results_list, USER_QUESTION))
print(answer)

 To get your daughter more interested in science, I would recommend getting her an age-appropriate science kit or set that allows for hands-on exploration and experimentation. For example, for a younger child you could try a beginner chemistry set, magnet set, or crystal growing kit. For an older child, look for kits that tackle more advanced scientific principles like physics, engineering, robotics, etc. The key is choosing something that sparks her natural curiosity and lets her actively investigate concepts through activities, observations, and discovery. Supplement the kits with science books, museum visits, documentaries, and conversations about science she encounters in everyday life. Making science fun and engaging is crucial for building her interest.
